#### Chapter 2

Problem: We have some images in the sequence missing. We have a 2-stage probem. Stage 1 - we use a seq-2-seq model to create the missing images. Stage 2 - We use latent AR process model to get the final densities.

Let's make the notes on preprocessing here. 

##### Stage 1

So we do the seq-2-seq model also on the train data we used to fine-tune the tasselnet model on China data.

Input seq length is None, 13, 32, and output shape should be None, 7, 32, where none denotes the batch size. Now first for each subwindow, we need to extract the 32 features. How to do this? - We can take the fine-tuned model and do the feature extraction - chop it off before the prediction head. Once we have it, we just have to stack it in the correct dimensions to have the None, 13, 32 shape for the input, and none, 7, 32 for the output.

There's a lot of overlap for the tasks here, maybe need to talk it out with Ghosh? 

Location of the original images and xml files seems to be:

block_0101 = '../../S_lab_TasselNet/Block_1_TN/Block_1_images_and_xml' etc etc as these are the ones used in preprocessing. Check if we really need these, or if we can get away with the existing preprocesed data. For thefirst task of finetuning the China model,we should be able to get away.

Also, the formal notes accompanying this work is in "Chapter_2_Dissertation_Notes" at overleaf.

Things to keep in mind:

1. The problem we are trying to solve: Some of the images that appear at the end of our sequence are missing. And we currenly assume these are the last 7 images of the sequence (we have 20 horizontal images per block, therefore we are doing a 13-7 split, we can change the splits for the seq-2-seq model if the model performance is too bad).

2. For latent AR process model, we used non-verlapping window sizes of 300, 300, 3 for feature extraction. But if we use this it will be too less data (only 48 as the train sample size, and 12 as the validation sample size), and it will not work well with our DL model.

3. Because of this, we will use may be 30, 30, 3 with a stride of 30 - still using overlapping data. Also, since we assume we do not have the later images, we need to be as accurate as possible with the extracted feature prediction, so lowering the window size, and making them overlapped will help (not using overlapping windows currently). 

4. Train a seq-2-seq model. See if the performance of this model could be improved by changing model architecture, seq-2-seq split (13-7), and overlapping nature, or the sub-window sizes.

5. Since we might need to experiment with the above parameters, make sure we right proper functions to quickly change the nature of the data that goes into the model. 

The data preprocessing script for train and validation data are in notebook "1_preprocessing_for_seq_2_seq_stage_1.ipynb", and the code for traning the seq2seq model is in the notebook "2_training_seq_2_seq_stage_1.ipynb". The codes for preprocessing the data for the other blocks (test data) are in the notebook "3_preprocessing_for_metrics_other_blocks_stage_1.ipynb". This notebook does preprocessing much faster than the methos in "1_preprocessing_for_seq_2_seq_stage_1.ipynb". The "4_Alternative_preprocessing_for_seq2seq_train_data_stage1.ipynb" notebook does sanity checks to make sure what we are doing in notebook 3 is correct.

We can now move onto the inference on the test data (All other blocks). Notebook "5_Inferenece_on_test_data_stage_1" does inference for the seq-to-seq model, taking all blocks separately, as well as together - need to verify this.

##### Stage 2

We have two types of models for this. One is a base model - a regular LSTM - time series model. And the other is out Bayesian Latent AR model.

###### Base model

We will first go forward with the LSTM sort of model. What is our story here? We have the features extracted for the last 7 sub-window sequence for each block. These are a set of sequential data. We need to use this sequence to predict the corresponding densities for each subwindow. Do we have the input sequences for this task stored anywhere? - yes, we have these in folder "seq_2_seq_test_data". But here is a problem. Do we use these 7? Or do we use the earlier 13 as the inputs? This is something to think about. In any case, we need the densities corresponding to the last 7 time periods for all (if not most) blocks. Let's preprocess the data for the densities first.

Before fitting this, let's look at the toy example from the DL with Python book - Chollet. Code in jupyter notebook 8. We see that the input features are actually in similar shape (batch_size, time_steps, features) as our baseline LSTM model that we need to fit, but in the case of their example, they have only a single output (tempertaure in 24 hours). We have the next 7 time points. Going by that logic, seems like we indeed require the time distribution layer in our model, but we can also fit a network with dense 7 to see if the model performances are very different for the two cases. 

###### Base model 1 - with time distribution layers

###### Base model 2 - with dense layer - 7 output neurons

These are implemented and evaluated in the notebooks 7 and 8.

###### Bayesian Latent AR process model

We have put this model on hold for a moment. We want accompany the suggestions of Dr. Ghosh, where we are starting off with the images as our inputs and extracted features as the targets, an alternative approach for the stage 1 implemented above (input extracted featrues and output extracted features). Since it will be a little messy to do everything in this folder, we are moving to the new folder CNN_se2seq_model in Spring 2025 main folder. Please refer that for implementation and the complementary details for this work.